### Cell 1 Configuration

In [48]:
import os
import torch

class Config:
    CSV_TRAIN = "train.csv"
    IMG_DIR_TRAIN = "data/train"
    
    # Model Settings
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  
    
    # Hyperparameters
    EPOCHS = 20
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    VAL_SPLIT_RATIO = 0.2
    RANDOM_SEED = 42
    
    # Hardware Device Detection
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {Config.DEVICE}")
if Config.DEVICE.type == 'cuda':
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

Using Device: cuda
🔥 GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Dataset and DataLoader

In [49]:
# CELL 2: DATASET & DATALOADERS (Using ImageFolder)

import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from dataset import IlluminationDataset, train_transforms, val_transforms

In [50]:
train_df = pd.read_csv(Config.CSV_TRAIN)
train_df.head()

,id,label
0,02137a86-0743-40e0-845b-6d22d1d5cc85,0
1,025d39a8-7859-4558-9bf9-bbdd475c6100,0
2,02a2a878-c5a4-490a-8061-6b2f4ac3b6d0,0
3,047f7996-9f0d-4a04-ae7f-24e246c407c7,0
4,052a9d62-a31f-4e4f-9a76-edac2d2ae95d,0


In [51]:
# 2. Split data into 80% training and 20% validation
train_df, val_df = train_test_split(
    train_df,
    test_size=Config.VAL_SPLIT_RATIO,
    random_state=Config.RANDOM_SEED,
    stratify=train_df["label"]
)

In [52]:
# 3. Create training and validation datasets
train_dataset = IlluminationDataset(
    dataframe=train_df,
    img_dir=Config.IMG_DIR_TRAIN,
    transform=train_transforms
)

val_dataset = IlluminationDataset(
    dataframe=val_df,
    img_dir=Config.IMG_DIR_TRAIN,
    transform=val_transforms
)

In [53]:
# 4. Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

In [54]:
# 5. Print dataset information
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_index())

Training images: 1200
Validation images: 300

Training class distribution:
label
0    400
1    400
2    400
Name: count, dtype: int64


### CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

In [55]:
# CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

import torch.nn as nn
import torchvision.models as models

def build_illumination_model():
    # 1. Load pretrained ResNet18
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # 2. Freeze all pretrained layers
    for param in model.parameters():
        param.requires_grad = False

    # 3. Unfreeze only the final residual block
    for param in model.layer4[1].parameters():
        param.requires_grad = True

    # 4. Replace the classification head
    num_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_features, 3)
    )

    return model


model = build_illumination_model().to(Config.DEVICE)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Model Initialized on {Config.DEVICE}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Model Initialized on cuda
Total Parameters: 11,178,051
Trainable Parameters: 4,722,179


#### Training on 8 million parameters

### CELL 4: TRAINING & VALIDATION ENGINE

In [56]:
from tqdm import tqdm # Great for visual progress bars in VS Code

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # Progress bar for the terminal/notebook
    loop = tqdm(dataloader, leave=False, desc="Training")
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        # 1. Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward Pass & Optimization
        
        loss.backward()
        optimizer.step()
        
        # 3. Track Metrics
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1) # Get the index of the highest logit
        correct_preds += torch.sum(preds == labels).item()
        total_samples += labels.size(0)
        
        # Update progress bar
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    with torch.no_grad(): # Disable gradient tracking for speed and memory
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

             # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Track metrics
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_preds += torch.sum(preds == labels).item()
            total_samples += labels.size(0)
            
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

In [57]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([1, 2, 0, 2, 2, 0, 0, 1, 2, 1])


### CELL 5: MAIN EXECUTION LOOP

In [58]:
import time
# 1. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()

# ONLY pass the trainable parameters to the optimizer
optimizer = torch.optim.AdamW(
    [
        {
            'params': model.layer4[1].parameters(),
            'lr': 1e-5
        },
        {
            'params': model.fc.parameters(),
            'lr': 1e-4
        }
    ],
    weight_decay=Config.WEIGHT_DECAY
)
# 1.5. Define Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

# 2. Initialize Tracking Variables
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🔥 Starting Training...")
start_time = time.time()

# 3. The Main Epoch Loop
for epoch in range(Config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
    print("-" * 20)
    
    # Train and Validate
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, Config.DEVICE)

    # Update learning rate based on validation loss
    scheduler.step(val_loss)
    
    # store metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Get current learning rate
    backbone_lr = optimizer.param_groups[0]['lr']
    fc_lr = optimizer.param_groups[1]['lr']

    print(f"Backbone LR: {backbone_lr:.8f}")
    print(f"FC LR:       {fc_lr:.8f}")
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
   
    
    # 4. Checkpointing Strategy (Save ONLY if Accuracy improves)
    if val_acc > best_val_acc:
        print(f"⭐ Validation Accuracy improved from {best_val_acc:.4f} to {val_acc:.4f}. Saving checkpoint!")

        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

# 5. Calculate total training time
time_elapsed = time.time() - start_time
print(
    f"\n✅ Training Complete in "
    f"{time_elapsed // 60:.0f}m "
    f"{time_elapsed % 60:.0f}s"
)
print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")

🔥 Starting Training...

Epoch 1/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 1.2452 | Train Acc: 0.3317
Val Loss:   1.1291 | Val Acc:   0.3667
⭐ Validation Accuracy improved from 0.0000 to 0.3667. Saving checkpoint!

Epoch 2/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 1.1833 | Train Acc: 0.3625
Val Loss:   1.1091 | Val Acc:   0.3700
⭐ Validation Accuracy improved from 0.3667 to 0.3700. Saving checkpoint!

Epoch 3/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 1.1555 | Train Acc: 0.3767
Val Loss:   1.0930 | Val Acc:   0.4033
⭐ Validation Accuracy improved from 0.3700 to 0.4033. Saving checkpoint!

Epoch 4/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 1.1213 | Train Acc: 0.4125
Val Loss:   1.0790 | Val Acc:   0.4233
⭐ Validation Accuracy improved from 0.4033 to 0.4233. Saving checkpoint!

Epoch 5/20
--------------------


...

Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 0.9225 | Train Acc: 0.5633
Val Loss:   1.0344 | Val Acc:   0.4733
⭐ Validation Accuracy improved from 0.4700 to 0.4733. Saving checkpoint!

Epoch 17/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 0.9358 | Train Acc: 0.5342
Val Loss:   1.0342 | Val Acc:   0.4800
⭐ Validation Accuracy improved from 0.4733 to 0.4800. Saving checkpoint!

Epoch 18/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 0.9359 | Train Acc: 0.5558
Val Loss:   1.0267 | Val Acc:   0.4900
⭐ Validation Accuracy improved from 0.4800 to 0.4900. Saving checkpoint!

Epoch 19/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 0.9125 | Train Acc: 0.5600
Val Loss:   1.0274 | Val Acc:   0.4833

Epoch 20/20
--------------------


Backbone LR: 0.00001000
FC LR:       0.00010000
Train Loss: 0.8972 | Train Acc: 0.5650
Val Loss:   1.0303 | Val Acc:   0.4733

✅ Training Complete in 4m 57s
🏆 Best Validation Accuracy: 0.4900


In [59]:
for epoch in range(len(history['train_loss'])):
    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Acc: {history['train_acc'][epoch]:.4f} | "
        f"Val Acc: {history['val_acc'][epoch]:.4f} | "
        f"Train Loss: {history['train_loss'][epoch]:.4f} | "
        f"Val Loss: {history['val_loss'][epoch]:.4f}"
    )

Epoch 01 | Train Acc: 0.3317 | Val Acc: 0.3667 | Train Loss: 1.2452 | Val Loss: 1.1291
Epoch 02 | Train Acc: 0.3625 | Val Acc: 0.3700 | Train Loss: 1.1833 | Val Loss: 1.1091
Epoch 03 | Train Acc: 0.3767 | Val Acc: 0.4033 | Train Loss: 1.1555 | Val Loss: 1.0930
Epoch 04 | Train Acc: 0.4125 | Val Acc: 0.4233 | Train Loss: 1.1213 | Val Loss: 1.0790
Epoch 05 | Train Acc: 0.4300 | Val Acc: 0.4333 | Train Loss: 1.1026 | Val Loss: 1.0743
Epoch 06 | Train Acc: 0.4383 | Val Acc: 0.4300 | Train Loss: 1.0912 | Val Loss: 1.0624
Epoch 07 | Train Acc: 0.4742 | Val Acc: 0.4400 | Train Loss: 1.0555 | Val Loss: 1.0599
Epoch 08 | Train Acc: 0.4567 | Val Acc: 0.4533 | Train Loss: 1.0495 | Val Loss: 1.0537
Epoch 09 | Train Acc: 0.4758 | Val Acc: 0.4500 | Train Loss: 1.0232 | Val Loss: 1.0488
Epoch 10 | Train Acc: 0.4792 | Val Acc: 0.4600 | Train Loss: 1.0181 | Val Loss: 1.0453
Epoch 11 | Train Acc: 0.4900 | Val Acc: 0.4500 | Train Loss: 1.0052 | Val Loss: 1.0450
Epoch 12 | Train Acc: 0.5225 | Val Acc: 0.4